In [ ]:
# 04_hybrid_lstm.ipynb

# 1. Imports
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.src_bak import data_loader, features, volatility, regimes, models_lstm

# 2. The Core Pipeline
# Re-running the clean, standard pipeline
raw_df = data_loader.fetch_raw_data(force_refresh=False)
stat_df = features.engineer_stationary_features(raw_df)

# ML-Safe Volatility Generation
garch_df = volatility.generate_expanding_garch(stat_df, window_size=252)

# Target Generation
regime_df = regimes.generate_smoothed_targets(
    garch_df, lower_quant=0.85, upper_quant=0.95, floor=0.0
)

# 3. Define the Features (No flat lags needed!)
# The LSTM will process these dynamically over the sequence length
core_features = [
    'Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
    'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff'
]

# 4. Train the Sequence Model
# We set sequence_length=21, which represents 1 trading month of history
model, scaler = models_lstm.train_and_evaluate_lstm(
    df=regime_df,
    feature_cols=core_features,
    target_col='Target_Smooth_10d',
    seq_length=21,    # Look back 21 days to predict day 22
    epochs=40,        # LSTMs take slightly longer to converge
    lr=0.001
)